# Session Demo Notebook
## Real-World EDA, Statistical Testing with Pingouin, SQL & Classification
### Dataset: UCI Heart Disease (Cleveland) — from Kaggle

---

**Kaggle Dataset:** https://www.kaggle.com/datasets/cherngs/heart-disease-cleveland-uci  
**Download:** `heart_cleveland_upload.csv`  

This is a **real clinical dataset** collected from the Cleveland Clinic. It has:
- Real missing values (coded as `?` in raw form)
- Skewed distributions (cholesterol, oldpeak)
- Outliers in resting blood pressure and cholesterol
- A binary classification target: `condition` (0 = no disease, 1 = disease)

---

### Column Reference
| Column | Description |
|--------|-------------|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Max heart rate achieved |
| `exang` | Exercise induced angina (1 = yes) |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels (0–3) |
| `thal` | Thalassemia (1 = normal, 2 = fixed defect, 3 = reversible defect) |
| `condition` | **Target**: 0 = no disease, 1 = disease |


---
## PART 0 — Setup


In [ ]:
!pip install pingouin

In [ ]:
# Install pingouin if not already available
# !pip install pingouin

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg
import sqlite3
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

print('All libraries loaded successfully ')

---
## PART 1 — Load & First Look

### Theory: Why real data is harder
> **Key insight for students:** The homework dataset was synthetically generated, so values were perfectly clean. Real clinical data:
> - Has genuine missingness (tests not always run)
> - Has biologically extreme values (outliers that are real, not errors)
> - Has skewed distributions (cholesterol is right-skewed in most populations)
>
> Your job as a data scientist is to **understand context** before deciding what to do.


In [ ]:
# !pip install opendatasets

In [ ]:
import opendatasets as od
od.download('https://www.kaggle.com/datasets/cherngs/heart-disease-cleveland-uci')
df = pd.read_csv('heart-disease-cleveland-uci/heart_cleveland_upload.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.ca.unique()

### Discussion Points
- Look at `chol` — max is 564 mg/dl. Is that medically possible? (Yes — rare but documented)
- Look at `trestbps` — is there a 0 value? Blood pressure of 0 is impossible → likely a data entry error
- Look at `ca` and `thal` — these may have been read as float due to missing values in the raw file

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

In [ ]:
# Visualise missing data pattern
# !pip install missingno  # uncomment if needed
# import missingno as msno
# msno.matrix(df)

# Alternative: heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap (yellow = missing)')
plt.tight_layout()
plt.show()

---
## PART 2 — Data Cleaning

### Theory: Types of Missingness (MCAR / MAR / MNAR)

| Type | Meaning | Implication |
|------|---------|-------------|
| **MCAR** | Missing Completely At Random | Safe to drop or impute with mean/median |
| **MAR** | Missing At Random (depends on other variables) | Impute using related variables |
| **MNAR** | Missing Not At Random (depends on the missing value itself) | Most dangerous — may need domain knowledge |



In [ ]:
# Check if missingness in 'ca' is related to the target
# This tests MAR: do patients with missing ca have different disease rates?
df['ca_missing'] = df['ca'].isnull().astype(int)

print("Disease rate when ca IS missing:")
print(df[df['ca_missing']==1]['condition'].value_counts(normalize=True))
print("\nDisease rate when ca is NOT missing:")
print(df[df['ca_missing']==0]['condition'].value_counts(normalize=True))

In [ ]:
# Decision: drop rows with missing values (only 6 rows — justifiable)
df_clean = df.drop(columns=['ca_missing']).dropna()
print(f'Shape after dropping missing rows: {df_clean.shape}')

# Also handle impossible values
print(f"\nRows with trestbps == 0: {(df_clean['trestbps'] == 0).sum()}")
df_clean = df_clean[df_clean['trestbps'] > 0]
print(f'Shape after removing invalid BP: {df_clean.shape}')

---
## PART 3 — Deeper EDA: Distributions, Skewness & Outliers

### Theory: Skewness
> - **Skewness = 0**: Perfectly symmetric (normal)
> - **Skewness > 1**: Right-skewed (long tail to the right) — common in income, cholesterol
> - **Skewness < -1**: Left-skewed (long tail to the left)
>
> Why does this matter? Models like Logistic Regression perform better with roughly normal inputs. Highly skewed features can also inflate the influence of outliers.


In [ ]:
# Compute skewness for all numerical columns
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
skewness = df_clean[numeric_cols].skew().round(3)
print("Skewness:")
print(skewness)
print("\n→ Values |skew| > 1 are highly skewed and may benefit from transformation.")

In [ ]:
# Distribution plots for all key numeric columns
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    sns.histplot(df_clean[col], kde=True, ax=ax, color='steelblue')
    ax.axvline(df_clean[col].mean(), color='red', linestyle='--', label='Mean')
    ax.axvline(df_clean[col].median(), color='orange', linestyle='--', label='Median')
    ax.set_title(f'{col} | skew={df_clean[col].skew():.2f}')
    ax.legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Distributions of Numerical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("\n Notice: 'oldpeak' is strongly right-skewed (many patients have 0, a few have high values)")
print(" 'chol' is also slightly right-skewed — common in real cholesterol data")

### Theory: Outlier Detection — IQR Method vs Z-Score

| Method | Formula | Best For |
|--------|---------|----------|
| **IQR** | < Q1 - 1.5×IQR or > Q3 + 1.5×IQR | Non-normal data, skewed |
| **Z-Score** | |z| > 3 | Normally distributed data |

> In clinical data, an outlier might be a **real extreme case**, not an error. Always investigate before removing!


In [ ]:
# Boxplots to visualize outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(['chol', 'trestbps', 'oldpeak']):
    sns.boxplot(y=df_clean[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(f'Outliers in {col}')
plt.suptitle('Box Plots — IQR Outlier Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Count outliers using IQR method
def count_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((series < lower) | (series > upper)).sum()
    return n_outliers, round(lower, 1), round(upper, 1)

for col in ['chol', 'trestbps', 'oldpeak']:
    n, lo, hi = count_outliers_iqr(df_clean[col])
    print(f"{col}: {n} outliers | acceptable range [{lo}, {hi}]")

In [ ]:
# Log-transform oldpeak (right-skewed)
# We use log1p because oldpeak can be 0
df_clean['oldpeak_log'] = np.log1p(df_clean['oldpeak'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_clean['oldpeak'], kde=True, ax=axes[0], color='coral')
axes[0].set_title(f'oldpeak — Original (skew={df_clean["oldpeak"].skew():.2f})')
sns.histplot(df_clean['oldpeak_log'], kde=True, ax=axes[1], color='seagreen')
axes[1].set_title(f'oldpeak — Log Transformed (skew={df_clean["oldpeak_log"].skew():.2f})')
plt.tight_layout()
plt.show()

---
## PART 4 — Statistical Testing with Pingouin

### Theory: Why We Use Statistical Tests
> Visualizations tell us what *might* be true. Statistical tests tell us whether what we observe is likely to be **real (statistically significant)** or just due to random chance.
>
> **Pingouin** is a Python library built on top of scipy/pandas that makes statistical tests:
> - More readable (returns DataFrames with full test results)
> - More complete (includes effect sizes, confidence intervals)
> - Easier to report


In [ ]:
import pingouin as pg

# Split cholesterol by disease condition
chol_disease = df_clean[df_clean['condition'] == 1]['chol']
chol_no_disease = df_clean[df_clean['condition'] == 0]['chol']

print(f"Mean cholesterol — No Disease: {chol_no_disease.mean():.1f}")
print(f"Mean cholesterol — Disease:    {chol_disease.mean():.1f}")

### 4.1 — Normality Test (Shapiro-Wilk)

> Before running a t-test, we check if the data is normally distributed.  
> **H₀**: The data is normally distributed.  
> If p < 0.05, we reject H₀ → data is NOT normal → use non-parametric test instead.


In [ ]:
# Normality test using Pingouin
print("=== Normality Test (Shapiro-Wilk) ===")
print("\nCholesterol — No Disease:")
print(pg.normality(chol_no_disease))
print("\nCholesterol — Disease:")
print(pg.normality(chol_disease))

### 4.2 — Chi-Square Test (Categorical vs Categorical)

> **Question**: Is chest pain type (`cp`) associated with heart disease (`condition`)?
>
> Chi-square tests whether two categorical variables are **independent**.  
> **H₀**: cp and condition are independent.  
> If p < 0.05, there IS a significant association.


In [ ]:
# Chi-square test
chi2_result = pg.chi2_independence(df_clean, x='cp', y='condition')
print("=== Chi-Square Test: Chest Pain Type vs Condition ===")
print(chi2_result[2].round(4))  # index 2 = main test stats

# Visualize
ct = pd.crosstab(df_clean['cp'], df_clean['condition'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, figsize=(8, 4), colormap='coolwarm')
plt.title('Chest Pain Type vs Disease (% within each cp type)')
plt.xlabel('Chest Pain Type')
plt.ylabel('Percentage')
plt.legend(['No Disease', 'Disease'], loc='upper right')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 4.3 — Pearson & Spearman Correlation (Pingouin)

> Pingouin's correlation gives you the coefficient, p-value, CI, and power — all in one line.


In [ ]:
# Pearson correlation: age vs thalach
print("=== Pearson Correlation: age vs thalach ===")
print(pg.corr(df_clean['age'], df_clean['thalach'], method='pearson').round(4))

print("\n=== Spearman Correlation: age vs thalach ===")
print(pg.corr(df_clean['age'], df_clean['thalach'], method='spearman').round(4))

In [ ]:
# Standard correlation heatmap
corr_matrix = df_clean[corr_cols].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Spearman Correlation Heatmap')
plt.tight_layout()
plt.show()

---
## PART 5 — SQL with SQLite

### Theory: Why SQL in a Data Science workflow?
> Most real data lives in relational databases, not CSV files. SQL lets you:
> - Filter, aggregate, join data before pulling into pandas
> - Reproduce queries reliably and efficiently
> - Communicate with data engineers
>
> In this demo, we load our DataFrame into an **in-memory SQLite database** — a common pattern for teaching SQL without a server.


In [ ]:
# Load dataframe into SQLite
conn = sqlite3.connect(':memory:')

df_clean.to_sql('patients', conn, index=False, if_exists='replace')

# Helper to run queries cleanly
def sql(query):
    return pd.read_sql_query(query, conn)

print('Database loaded ')
sql("SELECT * FROM patients LIMIT 5")

In [ ]:
# Basic aggregations
sql("""
SELECT
    condition,
    COUNT(*) AS n_patients,
    ROUND(AVG(age), 1) AS avg_age,
    ROUND(AVG(chol), 1) AS avg_chol,
    ROUND(AVG(thalach), 1) AS avg_max_hr,
    ROUND(AVG(trestbps), 1) AS avg_bp
FROM patients
GROUP BY condition
""")

In [ ]:
# Filter: high-risk patients (disease + high cholesterol)
sql("""
SELECT age, sex, cp, chol, thalach, condition
FROM patients
WHERE condition = 1
  AND chol > 250
ORDER BY chol DESC
LIMIT 10
""")

In [ ]:
# Window function: rank patients by cholesterol within each sex group
sql("""
SELECT
    age, sex, chol,
    RANK() OVER (PARTITION BY sex ORDER BY chol DESC) AS chol_rank_in_sex
FROM patients
LIMIT 15
""")

In [ ]:
# Subquery: patients with above-average cholesterol
sql("""
SELECT COUNT(*) AS n_above_avg_chol
FROM patients
WHERE chol > (SELECT AVG(chol) FROM patients)
""")

---
## PART 6 — Correlation & Feature Insights


In [ ]:
# Pairplot for key features, colored by condition
pair_cols = ['age', 'chol', 'thalach', 'oldpeak', 'condition']
g = sns.pairplot(df_clean[pair_cols], hue='condition', diag_kind='kde',
                 plot_kws={'alpha': 0.5}, palette='Set1')
g.fig.suptitle('Pairplot of Key Clinical Features by Condition', y=1.01)
plt.tight_layout()
plt.show()

print("Look for diagonal separation in any subplot — that variable is a strong discriminator")

## Pairplot and How to Analyze It

### What is a Pairplot?

A **pairplot** is a visualization that shows relationships between multiple numerical variables in a dataset. It plots every variable against every other variable in a grid format.

It helps in **exploratory data analysis (EDA)**.

---

### Structure of a Pairplot

- **Diagonal plots**  
  Show the distribution of each variable (histogram or KDE)

- **Off-diagonal plots**  
  Show scatter plots between pairs of variables

---

### How to Analyze a Pairplot

- **Check correlation direction**
  - Upward trend → positive correlation
  - Downward trend → negative correlation
  - Random scatter → no correlation

- **Check strength of relationship**
  - Tight pattern → strong relationship
  - Scattered points → weak relationship

- **Identify linear vs non-linear patterns**
  - Straight line → linear relationship
  - Curve → non-linear relationship
  - No clear pattern → weak/no relationship

- **Look for clusters**
  - Grouped points may indicate hidden classes or patterns

- **Detect outliers**
  - Isolated points far from others may affect analysis

---

### Why Pairplots are Useful

- Helps understand relationships between variables
- Detects multicollinearity
- Identifies patterns and trends
- Useful for early data exploration

---

### Summary

A pairplot provides a quick visual overview of:
- Distributions of variables
- Relationships between features
- Patterns, clusters, and outliers in the dataset

---
## PART 7 — Classification Model: Logistic Regression

### Theory: Logistic Regression
> Logistic Regression is the **simplest classification model**. It:
> - Estimates the **probability** that a sample belongs to a class
> - Uses a sigmoid function to map linear output → [0, 1]
> - Decision boundary: if P > 0.5 → predict class 1
>
> **Why start here?**
> - Interpretable (coefficients have meaning)
> - Fast
> - Good baseline before trying complex models
> - Coefficients reveal which features matter


In [ ]:
# Prepare features
feature_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                'restecg', 'thalach', 'exang', 'oldpeak', 'slope']

X = df_clean[feature_cols].copy()
y = df_clean['condition']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'\nClass distribution in train set:')
print(y_train.value_counts(normalize=True).round(3))

In [ ]:
# Fit Logistic Regression
from sklearn.preprocessing import StandardScaler

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

print('Model trained')

In [ ]:
# Evaluation
print("=" * 50)
print("CLASSIFICATION REPORT")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

In medicine, the goal is often to catch as many actual cases as possible, even if it means a few false alarms. If a model predicts someone is healthy when they are actually sick (false negative), it can be dangerous.

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Feature Importance (coefficients)
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=True)

colors = ['salmon' if c < 0 else 'steelblue' for c in coef_df['coefficient']]
axes[1].barh(coef_df['feature'], coef_df['coefficient'], color=colors)
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Logistic Regression Coefficients\n(blue = increases risk, red = decreases risk)')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.show()

### Interpreting Coefficients
> - **Positive coefficient** → higher value of that feature → higher probability of disease
> - **Negative coefficient** → higher value → lower probability of disease
> - **Magnitude** → how strongly the feature influences the prediction
>
> `cp` (chest pain type) typically has a strong positive coefficient — asymptomatic chest pain is counterintuitively associated with disease.
